# SAFE-VISION: Datasets and Multimodal Large Language Models (MLLMs) Setup

This consolidated notebook contains both the open-source dataset definitions and the loading code for open-source Multimodal Large Language Models (MLLMs) extracted from the research papers. 

## Notebook Structure:
1. **Part 1: Open-Source Datasets** (Metadata, modalities, and custom loading functions for 16 datasets)
2. **Part 2: Open-Source MLLM Models** (Boilerplate code to import and load models via Hugging Face `transformers` and PyTorch)

# Part 1: Open-Source Datasets

We have cataloged 16 key open-source datasets related to multimodal content moderation, video/audio safety, and hateful memes.

In [ ]:
import os
import urllib.request
from typing import Dict, Any, Optional

# Dict containing metadata for all extracted datasets
DATASETS_METADATA: Dict[str, Dict[str, Any]] = {
    # --- Video / Audio / Audio-Visual Datasets ---
    "safewatch_bench": {
        "name": "SAFEWATCH-BENCH",
        "has_video": True,
        "has_audio": False,
        "modality": "Video",
        "description": "Large-scale video guardrail dataset containing 2M video clips (Real-world and GenAI splits) across 6 unsafe categories.",
        "source_paper": "Safewatch (ICLR 2025)",
        "access_link": "https://github.com/ICLR2025-Safewatch/Safewatch",
        "huggingface_id": None
    },
    "kuaimod": {
        "name": "KuaiMod",
        "has_video": True,
        "has_audio": True,
        "modality": "Audio-Visual (Video + Audio)",
        "description": "Short Video Platform (SVP) content moderation benchmark from Kuaishou, with 24,562 video samples covering 15 categories of violations.",
        "source_paper": "KuaiMod SVP Governance",
        "access_link": "https://github.com/Kuaishou-Research/KuaiMod",
        "huggingface_id": None
    },
    "xd_violence": {
        "name": "XD-Violence",
        "has_video": True,
        "has_audio": True,
        "modality": "Audio-Visual (Video + Audio)",
        "description": "A large-scale video dataset for violence detection in both video and audio streams (explosions, gunshots, screams). Contains 4,754 videos.",
        "source_paper": "XD-Violence (ECCV 2020)",
        "access_link": "https://roc-ng.github.io/XD-Violence/",
        "huggingface_id": "detection-datasets/xd-violence"
    },
    "ucf_crime": {
        "name": "UCF-Crime",
        "has_video": True,
        "has_audio": False,
        "modality": "Video",
        "description": "Surveillance videos capturing real-world anomalies, crimes, and safety hazards, used for anomaly detection.",
        "source_paper": "Real-world Anomaly Detection in Surveillance Videos (CVPR 2018)",
        "access_link": "https://www.crcv.ucf.edu/research/projects/real-world-anomaly-detection-in-surveillance-videos/",
        "huggingface_id": None
    },
    "fakesv": {
        "name": "FakeSV",
        "has_video": True,
        "has_audio": True,
        "modality": "Audio-Visual + Text",
        "description": "A multimodal benchmark for fake news detection on short videos, including rich social context, visual content, and audio cues.",
        "source_paper": "FakeSV (AAAI 2023 / ACM MM)",
        "access_link": "https://github.com/FakeSV/FakeSV-Benchmark",
        "huggingface_id": None
    },
    "autoshot": {
        "name": "Autoshot",
        "has_video": True,
        "has_audio": False,
        "modality": "Video",
        "description": "A short video dataset specifically compiled for Shot Boundary Detection (SBD) to analyze scene transitions.",
        "source_paper": "Autoshot: A Short Video Dataset",
        "access_link": "https://github.com/AutoShot-SBD/AutoShot",
        "huggingface_id": None
    },
    "vhd11k": {
        "name": "VHD11K",
        "has_video": True,
        "has_audio": False,
        "modality": "Video",
        "description": "Video Harmfulness Recognition dataset comprising 11,000 video samples for toxic and harmful visual content filtering.",
        "source_paper": "Video Harmfulness Recognition Benchmark",
        "access_link": "https://github.com/VHD11K/VHD11K",
        "huggingface_id": None
    },
    "vsd": {
        "name": "Violent Scenes Dataset (VSD)",
        "has_video": True,
        "has_audio": True,
        "modality": "Audio-Visual (Video + Audio)",
        "description": "Dataset containing movie scenes labeled for violence, capturing visual actions and acoustic indices like explosions or screaming.",
        "source_paper": "The Violent Scenes Dataset (VSD)",
        "access_link": "https://www.interdigital.com/research-innovation/technologies/multimedia/vsd-dataset",
        "huggingface_id": None
    },
    "blm_guard": {
        "name": "BLM-Guard Benchmark",
        "has_video": True,
        "has_audio": False,
        "modality": "Video",
        "description": "A real-world commercial short-video ads dataset for ad moderation, structured across seven risk tiers.",
        "source_paper": "BLM-Guard: Safeguarding Vision Curation (AAAI 2026)",
        "access_link": "https://github.com/YangY-PHI/BLM-Guard",
        "huggingface_id": None
    },
    "lspd": {
        "name": "LSPD (Large-scale Pornographic Dataset)",
        "has_video": True,
        "has_audio": False,
        "modality": "Video / Image",
        "description": "Large-scale pornographic dataset for detection, classification, and age-appropriate content management systems.",
        "source_paper": "LSPD: Large-Scale Pornographic Dataset",
        "access_link": "https://github.com/Phan-et-al/LSPD",
        "huggingface_id": None
    },
    # --- Multimodal Meme (Image-Text) Datasets ---
    "facebook_hateful_memes": {
        "name": "Facebook Hateful Memes (FHM)",
        "has_video": False,
        "has_audio": False,
        "modality": "Image-Text Meme",
        "description": "A multimodal dataset consisting of 10,000+ memes, specifically designed to test visual-textual hate speech detection.",
        "source_paper": "The Hateful Memes Challenge (NeurIPS 2020)",
        "access_link": "https://ai.meta.com/tools/hatefulmemes/",
        "huggingface_id": "facebook/hateful_memes"
    },
    "harmeme": {
        "name": "HarMeme",
        "has_video": False,
        "has_audio": False,
        "modality": "Image-Text Meme",
        "description": "A repository of harmful memes (original memes) annotated for severity and harm potential.",
        "source_paper": "HarMeme: Multimodal Harmful Meme Detection",
        "access_link": "https://github.com/LCS2-IIITD/HarMeme",
        "huggingface_id": None
    },
    "mami": {
        "name": "MAMI (Multimodal Abuse Detection)",
        "has_video": False,
        "has_audio": False,
        "modality": "Image-Text Meme",
        "description": "Multimodal Abuse detection against Women on Instagram meme dataset, capturing misogyny.",
        "source_paper": "SemEval-2022 Task 5: Multimodal Misogyny Detection",
        "access_link": "https://competitions.codalab.org/competitions/34175",
        "huggingface_id": "semeval2022_task5"
    },
    "hatred": {
        "name": "HatReD (Hateful meme with Reasons Dataset)",
        "has_video": False,
        "has_audio": False,
        "modality": "Image-Text Meme + Text Reasons",
        "description": "An extension of the Facebook Hateful Memes (FHM) dataset that includes additional human-annotated explanation reasons.",
        "source_paper": "Hateful Memes with Reasons Dataset (NeurIPS/ICLR workshops)",
        "access_link": "https://github.com/HatReD-dataset/HatReD",
        "huggingface_id": None
    },
    "multioff": {
        "name": "MultiOFF",
        "has_video": False,
        "has_audio": False,
        "modality": "Image-Text Meme",
        "description": "Multimodal meme dataset for identifying offensive content on social media.",
        "source_paper": "MultiOFF: Multimodal Meme Dataset",
        "access_link": "https://github.com/smartdata-cs-unibo/MultiOFF",
        "huggingface_id": None
    },
    # --- Text-Only Datasets ---
    "toxigen": {
        "name": "Toxigen",
        "has_video": False,
        "has_audio": False,
        "modality": "Text",
        "description": "Large-scale machine-generated dataset for implicit and adversarial hate speech detection.",
        "source_paper": "Toxigen (ACL 2022)",
        "access_link": "https://github.com/microsoft/TOXIGEN",
        "huggingface_id": "microsoft/toxigen"
    }
}

In [ ]:
def list_datasets() -> None:
    """Prints out all available datasets with their modalities and descriptions."""
    print("=" * 80)
    print(f"{'AVAILABLE DATASETS':^80}")
    print("=" * 80)
    for key, data in DATASETS_METADATA.items():
        print(f"Key: {key:<20} | Name: {data['name']}")
        print(f"Modality: {data['modality']}")
        print(f"Video Support: {'YES' if data['has_video'] else 'NO'} | Audio Support: {'YES' if data['has_audio'] else 'NO'}")
        print(f"Description: {data['description']}")
        print(f"Access/Link: {data['access_link']}")
        print("-" * 80)

def load_dataset(dataset_key: str) -> Optional[Any]:
    """
    Attempts to import and load the dataset.
    For Hugging Face datasets (e.g., FHM, MAMI, Toxigen), it will try to use the HF 'datasets' library.
    For local or external video/audio datasets, it prints out download and setup instructions.
    """
    if dataset_key not in DATASETS_METADATA:
        print(f"Error: Dataset '{dataset_key}' is not registered in the system.")
        return None
        
    meta = DATASETS_METADATA[dataset_key]
    print(f"\n[INFO] Loading Dataset: {meta['name']} ({meta['modality']})")
    
    # Audio/Video classification
    if meta["has_video"] and meta["has_audio"]:
        print(">> Note: This is an Audio-Visual dataset (contains both video and audio streams).")
    elif meta["has_video"]:
        print(">> Note: This is a Video dataset (contains visual frames).")
    elif meta["has_audio"]:
        print(">> Note: This is an Audio dataset (contains audio waveforms).")
    else:
        print(">> Note: This is a Non-AV dataset (Memes, Images, or Text).")
        
    hf_id = meta["huggingface_id"]
    if hf_id:
        try:
            print(f"Attempting to load '{hf_id}' via Hugging Face...")
            from datasets import load_dataset as hf_load_dataset
            dataset = hf_load_dataset(hf_id)
            print(f"Successfully loaded {meta['name']} via Hugging Face!")
            return dataset
        except ImportError:
            print("Hugging Face 'datasets' library is not installed. Run: pip install datasets")
            print(f"Access dataset manually at: {meta['access_link']}")
            return None
        except Exception as e:
            print(f"Failed to load: {e}. Download manually at: {meta['access_link']}")
            return None
    else:
        print("This dataset is not hosted on Hugging Face (or requires manual registration).")
        print(f"Please download and configure from: {meta['access_link']}")
        return None

# List all datasets
list_datasets()

# Part 2: Open-Source MLLM Models

We import and provide loading templates for the open-source Multimodal Large Language Models (MLLMs) and Vision-Language Models (VLMs) used in the research papers.

In [ ]:
# Check CUDA GPU acceleration
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Count:", torch.cuda.device_count())
    print("Current Device Name:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Models will load in CPU mode (Warning: may be slow or run out of RAM).")

### 1. Qwen2-VL
**Qwen2-VL** is Alibaba's state-of-the-art open-source MLLM series, with top-tier capabilities in video and visual document understanding.

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

def load_qwen2_vl(model_id="Qwen/Qwen2-VL-7B-Instruct"):
    print(f"Loading {model_id}...")
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )
    processor = AutoProcessor.from_pretrained(model_id)
    print("Qwen2-VL successfully loaded.")
    return model, processor

# To load, uncomment:
# model_qwen, processor_qwen = load_qwen2_vl()

### 2. InternVL2
**InternVL2** is an extremely strong open-source vision-language model family by OpenGVLab.

In [ ]:
from transformers import AutoModel, AutoTokenizer

def load_internvl2(model_id="OpenGVLab/InternVL2-8B"):
    print(f"Loading {model_id}...")
    # InternVL2 has custom model definitions, requiring trust_remote_code=True
    model = AutoModel.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
        device_map="auto"
    ).eval()
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True, use_fast=False)
    print("InternVL2 successfully loaded.")
    return model, tokenizer

# To load, uncomment:
# model_intern, tokenizer_intern = load_internvl2()

### 3. LLaVA-NeXT-Video
**LLaVA-NeXT-Video** is optimized specifically to process video frames dynamically over time.

In [ ]:
from transformers import LlavaNextVideoForConditionalGeneration, LlavaNextVideoProcessor

def load_llava_next_video(model_id="llava-hf/LLaVA-NeXT-Video-7B-hf"):
    print(f"Loading {model_id}...")
    model = LlavaNextVideoForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )
    processor = LlavaNextVideoProcessor.from_pretrained(model_id)
    print("LLaVA-NeXT-Video successfully loaded.")
    return model, processor

# To load, uncomment:
# model_llava_video, processor_llava_video = load_llava_next_video()

### 4. Llama-Guard-3-11B-Vision
**Llama-Guard-3-11B-Vision** is fine-tuned to classify multimodal prompts/outputs as safe or unsafe against standard safety taxonomies.

In [ ]:
from transformers import MllamaForConditionalGeneration, AutoProcessor

def load_llama_guard_3_vision(model_id="meta-llama/Llama-Guard-3-11B-Vision"):
    print(f"Loading {model_id}...")
    # Note: Requires HF token login and gated repository access
    model = MllamaForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )
    processor = AutoProcessor.from_pretrained(model_id)
    print("Llama-Guard-3-Vision successfully loaded.")
    return model, processor

# To load, uncomment:
# model_guard, processor_guard = load_llama_guard_3_vision()

### 5. MiniCPM-V-2.6
**MiniCPM-V-2.6** is a lightweight VLM with excellent OCR and video capabilities, designed for edge/mobile and lower-spec GPUs.

In [ ]:
def load_minicpm_v26(model_id="openbmb/MiniCPM-V-2_6"):
    print(f"Loading {model_id}...")
    model = AutoModel.from_pretrained(
        model_id,
        trust_remote_code=True,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    ).eval()
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    print("MiniCPM-V-2.6 successfully loaded.")
    return model, tokenizer

# To load, uncomment:
# model_minicpm, tokenizer_minicpm = load_minicpm_v26()

### 6. InstructBLIP & BLIP-2
Baseline visual-question answering and image-to-text models.

In [ ]:
from transformers import InstructBlipForConditionalGeneration, InstructBlipProcessor

def load_instruct_blip(model_id="Salesforce/instructblip-vicuna-7b"):
    print(f"Loading {model_id}...")
    model = InstructBlipForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )
    processor = InstructBlipProcessor.from_pretrained(model_id)
    print("InstructBLIP successfully loaded.")
    return model, processor

# To load, uncomment:
# model_blip, processor_blip = load_instruct_blip()